In [ ]:
%load_ext autoreload
%autoreload 2

from source.dataset import GraphDataset, CombinedDataset, GraphDataModule
from source.model import GravNetModel


In [ ]:
import glob
import tqdm
import torch
from torch_geometric.data import DataLoader
import torch.nn as nn 
import torch.nn.functional as F
# from source.train import GravNetLightning
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import pytorch_lightning as pl
from pytorch_lightning import Trainer
import os
from source.preprocess import preprocess_data

In [ ]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10000/*.root")
# preprocess_data(root_files, "data/pt/10000/")

In [ ]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10001/*.root")
# preprocess_data(root_files, "data/pt/10001/")

In [ ]:
root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10002/*.root")
preprocess_data(root_files, "data/pt/10002/")

In [ ]:
# root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10003/*.root")
# preprocess_data(root_files, "data/pt/10003/")

In [41]:
pt = torch.load("/home/benwilson/work/nuGraph/data/pt/10003/10003_000.pt")
for event in pt:
    print(event)

/tmp/ipykernel_2555998/2704265008.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pt = torch.load("/home/benwilson/work/nuGraph/data/pt/10003/10003_000.pt")


Data(x=[9278, 3], y_class=3, y_energy=6.067638397216797)
Data(x=[41337, 3], y_class=3, y_energy=7.162242412567139)
Data(x=[6463, 3], y_class=3, y_energy=5.602155685424805)
Data(x=[18582, 3], y_class=3, y_energy=6.451496601104736)
Data(x=[651, 3], y_class=3, y_energy=5.294209003448486)
Data(x=[5768, 3], y_class=3, y_energy=6.072768211364746)
Data(x=[11190, 3], y_class=3, y_energy=6.399509429931641)
Data(x=[4374, 3], y_class=3, y_energy=5.968733310699463)
Data(x=[73868, 3], y_class=3, y_energy=7.562889099121094)
Data(x=[837, 3], y_class=3, y_energy=5.592515468597412)
Data(x=[9332, 3], y_class=3, y_energy=5.769757270812988)
Data(x=[10920, 3], y_class=3, y_energy=7.363089561462402)
Data(x=[1009, 3], y_class=3, y_energy=6.6440110206604)
Data(x=[4097, 3], y_class=3, y_energy=5.97335147857666)
Data(x=[56638, 3], y_class=3, y_energy=7.514309406280518)
Data(x=[878, 3], y_class=3, y_energy=4.886356353759766)
Data(x=[63051, 3], y_class=3, y_energy=7.951524257659912)
Data(x=[11113, 3], y_class=3, 

In [ ]:
class GravNetLightning(pl.LightningModule):
    def __init__(
        self,
        lr=1e-3,
        lambda_reg=0.5,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.model = GravNetModel()

        self.cls_loss = nn.CrossEntropyLoss()
        self.reg_loss = nn.HuberLoss()

    def forward(self, data):
        return self.model(data)

    def training_step(self, batch, batch_idx):
        
        if batch_idx % 50 == 0:
            print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")


        class_logits, energy_pred = self(batch)

        cls_loss = self.cls_loss(class_logits, batch.y_class)
        reg_loss = self.reg_loss(energy_pred, batch.y_energy)

        loss = cls_loss + self.hparams.lambda_reg * reg_loss

        preds = torch.argmax(class_logits, dim=1)
        acc = (preds == batch.y_class).float().mean()

        self.log("train_loss", loss.detach(), prog_bar=True, batch_size=batch.num_graphs)
        self.log("train_cls_loss", cls_loss.detach(), batch_size=batch.num_graphs)
        self.log("train_reg_loss", reg_loss.detach(), batch_size=batch.num_graphs)
        self.log("train_acc", acc.detach(), prog_bar=True, batch_size=batch.num_graphs)


        return loss

    def validation_step(self, batch, batch_idx):
        class_logits, energy_pred = self(batch)

        cls_loss = self.cls_loss(class_logits, batch.y_class)
        reg_loss = self.reg_loss(energy_pred, batch.y_energy)

        loss = cls_loss + self.hparams.lambda_reg * reg_loss

        preds = torch.argmax(class_logits, dim=1)
        acc = (preds == batch.y_class).float().mean()

        self.log("val_loss", loss.detach(), prog_bar=True, batch_size=batch.num_graphs)
        self.log("val_acc", acc.detach(), prog_bar=True, batch_size=batch.num_graphs)


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=50,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler,
        }




In [ ]:
model = GravNetLightning()

# logger = TensorBoardLogger()
        # save_dir=cfg.logging.log_dir,
        # save_dir=hydra.core.hydra_config.HydraConfig.get().runtime.output_dir,    )

pt_files = glob.glob("data/pt/*/*.pt")
assert len(pt_files) > 0, "No .pt files found" 

datamodule = GraphDataModule(
    pt_files=pt_files,
    batch_size=1,
    num_workers=1,
)


checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min",
)  

early_stop_cb = EarlyStopping(
monitor="val_loss",
min_delta=0,
patience=12,
mode="min",
verbose=True,
)


trainer = Trainer(
    max_epochs=1000,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb],
    log_every_n_steps=50,
    accumulate_grad_batches=1
)

trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)
